# Hugging Face Fundamentals — Lesson 6: Model Inference

> Learning material for **Hugging Face Fundamentals**. Companion to the lesson script `06_Model_Inference.py` (same content, runnable without Jupyter).

**Task ID:** HF-006  |  **Folder:** `06_Model_Inference`


## The recipe behind every prediction

Last lesson we saw the pieces. Now put them together by hand — the pipeline (HF-002) does exactly these four steps:

1. **tokenize** — text → ids,
2. **forward** — ids → logits (raw scores),
3. **softmax** — logits → probabilities (they sum to 1),
4. **argmax** — probabilities → best class.

**Step 1+2 — tokenize and forward:**


In [ ]:
import torch
from transformers import AutoModelForSequenceClassification, AutoTokenizer

model_id = "distilbert/distilbert-base-uncased-finetuned-sst-2-english"
tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForSequenceClassification.from_pretrained(model_id)
model.eval()   # inference mode: no dropout

text = "Hugging Face is awesome!"
enc = tokenizer(text, return_tensors="pt")
with torch.no_grad():
    logits = model(**enc).logits
print("logits:", logits.tolist())


Logits are **raw** scores — they can be any number, even negative. They are not probabilities yet.

**Step 3 — softmax turns them into probabilities:**


In [ ]:
probs = torch.softmax(logits, dim=-1)
print("probs :", probs.tolist(), "sum =", probs.sum().item())

neg, pos = probs[0].tolist()
print(f"P(NEGATIVE) = {neg:.4f}")
print(f"P(POSITIVE) = {pos:.4f}")


**Step 4 — argmax picks the winner:**


In [ ]:
best = int(probs.argmax().item())
label = model.config.id2label[best]
print(f"prediction: {label} with {probs[0][best]:.3f}")


## Verify: the pipeline does the same math

Same model, same text — the numbers must match:


In [ ]:
from transformers import pipeline

pipe = pipeline("text-classification", model=model_id)
r = pipe(text, top_k=None)[0]
print("pipeline:", r["label"], round(r["score"], 4))
print("manual  :", label, round(probs[0][best].item(), 4))


## Why batched inputs are matrices

Models process a *batch* of texts at once (that is why we pad). Run the same text set through the loop for all texts and you get aprobability table:


In [ ]:
texts = ["Hugging Face is awesome!", "terrible experience", "it was okay"]
enc = tokenizer(texts, padding=True, truncation=True, return_tensors="pt")
with torch.no_grad():
    probs = torch.softmax(model(**enc).logits, dim=-1)
for t, p in zip(texts, probs):
    lbl = model.config.id2label[int(p.argmax())]
    print(f"{lbl:<9} ({p.max():.2f})  {t!r}")


## Try it yourself

1. Classify 5 of your own sentences with the manual loop.
2. Find a text where `P` sits near 0.5 — the model is unsure. Why?
3. Print the logits before and after softmax — see the difference.

## Common pitfalls

- **Forgot `model.eval()` / `no_grad()`** — dropout noise and wasted memory.
- **Softmax over the wrong dimension** — use `dim=-1` (per row).
- **`id2label` missing on some models** — fall back to your own map.

## Summary

- Inference = tokenize → forward → softmax → argmax.
- Logits (any sign) → probabilities (sum 1) → class id.
- The pipeline is this recipe, packaged.

**Next lesson:** HF-007 — Loading Pretrained Models.  |  Extra reading: `../resources/reference_links.md`
